# Caspase and Viability Data Cleaning & Merging

This notebook performs comprehensive data cleaning and merging for all caspase assays:

## Data Sources
1. **NCATS 13 Cell Types** (AID 654-667): Caspase-3/7 activation assays across 13 cell lines
2. **Tox21 Caspase** (AID 1346978 -hepg2, 1346980- chook1)

## Merge Logic
- Remove rows with missing CID
- **Keep CID with different outcome values** (Active, Inactive, Inconclusive)
- **If same CID + same outcome**: Keep the record with **highest activity score**

In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

print("Libraries loaded successfully!")

Libraries loaded successfully!


## Configuration: Define All Caspase Assays

NCATS assays map to different cell lines (based on PubChem assay descriptions)

In [ ]:

ASSAY_METADATA = {
    # Tox21 Caspase-3/7 Assays
    '1346978': {
        'assay_name': 'AID_1346978_caspase37_HepG2',
        'assay_type': 'caspase',
        'cell_line': 'HepG2',
        'source': 'Tox21',
        'is_counter_screen': False,
        'biological_process': 'apoptosis',
        'endpoint_level': 'mechanistic'
    },
    '1346980': {
        'assay_name': 'AID_1346980_caspase37_CHO-K1',
        'assay_type': 'caspase',
        'cell_line': 'CHO-K1',
        'source': 'Tox21',
        'is_counter_screen': False,
        'biological_process': 'apoptosis',
        'endpoint_level': 'mechanistic'
    },
    # NCATS Caspase-3 Assays (13 cell lines)
    '654': {'assay_name': 'AID_654_caspase3_HepG2', 'assay_type': 'caspase', 'cell_line': 'HepG2', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '655': {'assay_name': 'AID_655_caspase3_Jurkat', 'assay_type': 'caspase', 'cell_line': 'Jurkat', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '656': {'assay_name': 'AID_656_caspase3_HUV-EC-C', 'assay_type': 'caspase', 'cell_line': 'HUV-EC-C', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '657': {'assay_name': 'AID_657_caspase3_SHSY5Y', 'assay_type': 'caspase', 'cell_line': 'SHSY5Y', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '658': {'assay_name': 'AID_658_caspase3_BJ', 'assay_type': 'caspase', 'cell_line': 'BJ', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '659': {'assay_name': 'AID_659_caspase3_MRC-5', 'assay_type': 'caspase', 'cell_line': 'MRC-5', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '660': {'assay_name': 'AID_660_caspase3_Mesangial', 'assay_type': 'caspase', 'cell_line': 'Mesangial', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '661': {'assay_name': 'AID_661_caspase3_SK-N-SH', 'assay_type': 'caspase', 'cell_line': 'SK-N-SH', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '663': {'assay_name': 'AID_663_caspase3_H-4-II-E', 'assay_type': 'caspase', 'cell_line': 'H-4-II-E', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '664': {'assay_name': 'AID_664_caspase3_Hek293', 'assay_type': 'caspase', 'cell_line': 'Hek293', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '665': {'assay_name': 'AID_665_caspase3_N2a', 'assay_type': 'caspase', 'cell_line': 'N2a', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '666': {'assay_name': 'AID_666_caspase3_NIH 3T3', 'assay_type': 'caspase', 'cell_line': 'NIH 3T3', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
    '667': {'assay_name': 'AID_667_caspase3_RenalProximalTubule', 'assay_type': 'caspase', 'cell_line': 'Renal_Proximal_Tubule', 'source': 'NCATS', 'is_counter_screen': False, 'biological_process': 'apoptosis', 'endpoint_level': 'mechanistic'},
}

# Data paths (relative to notebook location)
NCATS_PATH = '../../Data/ncats_13_cell_types'
TOX21_PATH = '../../Data/tox21_caspase'

print(f"Defined metadata for {len(ASSAY_METADATA)} assays.")
print(f"\nCell lines:")
for aid, meta in ASSAY_METADATA.items():
    print(f"  AID {aid}: {meta['cell_line']} ({meta['source']})")

---
## Section 1: Load and Clean NCATS 13 Cell Types Data

In [3]:
def load_and_clean_assay(aid, metadata, is_tox21=False):
    """
    Load and clean a single assay file.
    """
    if is_tox21:
        file_path = f"{TOX21_PATH}/AID_{aid}_datatable.csv"
        skiprows = [1, 2, 3, 4, 5]
    else:
        file_path = f"{NCATS_PATH}/AID_{aid}_datatable.csv"
        skiprows = [1, 2, 3, 4]
    
    if not os.path.exists(file_path):
        print(f"  WARNING: File not found: {file_path}")
        return None
    
    # Load data - skip metadata rows
    df = pd.read_csv(file_path, skiprows=skiprows, low_memory=False)
    initial_rows = len(df)
    
    # Clean CID column
    df['PUBCHEM_CID'] = pd.to_numeric(df['PUBCHEM_CID'], errors='coerce')
    missing_cid = df['PUBCHEM_CID'].isna().sum()
    df = df.dropna(subset=['PUBCHEM_CID'])
    df['PUBCHEM_CID'] = df['PUBCHEM_CID'].astype(int)
    
    # Clean activity score
    df['PUBCHEM_ACTIVITY_SCORE'] = pd.to_numeric(df['PUBCHEM_ACTIVITY_SCORE'], errors='coerce').fillna(0)
    
    # Add assay metadata
    df['AID'] = aid
    df['assay_name'] = metadata['assay_name']
    df['assay_type'] = metadata['assay_type']
    df['cell_line'] = metadata['cell_line']
    df['source'] = metadata['source']
    df['is_counter_screen'] = metadata['is_counter_screen']
    df['biological_process'] = metadata['biological_process']
    df['endpoint_level'] = metadata['endpoint_level']
    
    final_rows = len(df)
    print(f"  AID {aid:>7} ({metadata['cell_line']:20s}): {initial_rows:>6,} -> {final_rows:>6,} rows (removed {missing_cid} missing CIDs)")
    
    return df

# Load NCATS assays
print("Loading NCATS Caspase-3 Assays (13 cell lines)...")
print("="*70)

all_dfs = []
ncats_aids = ['654', '655', '656', '657', '658', '659', '660', '661', '663', '664', '665', '666', '667']
for aid in ncats_aids:
    if aid in ASSAY_METADATA:
        df = load_and_clean_assay(aid, ASSAY_METADATA[aid], is_tox21=False)
        if df is not None:
            all_dfs.append(df)

print("="*70)
print(f"\nLoaded {len([d for d in all_dfs if d['source'].iloc[0] == 'NCATS'])} NCATS assays")

Loading NCATS Caspase-3 Assays (13 cell lines)...
  AID     654 (HepG2               ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     655 (Jurkat              ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     656 (HUV-EC-C            ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     657 (SHSY5Y              ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     658 (BJ                  ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     659 (MRC-5               ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     660 (Mesangial           ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     661 (SK-N-SH             ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     663 (H-4-II-E            ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     664 (Hek293              ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AID     665 (N2a                 ):  1,407 ->  1,393 rows (removed 14 missing CIDs)
  AI

In [4]:
# Summary of loaded data by cell line
print("\nData Summary by Cell Line:")
print("-"*70)
for df in all_dfs:
    aid = df['AID'].iloc[0]
    cell = df['cell_line'].iloc[0]
    source = df['source'].iloc[0]
    n_rows = len(df)
    n_cids = df['PUBCHEM_CID'].nunique()
    outcomes = df['PUBCHEM_ACTIVITY_OUTCOME'].value_counts().to_dict()
    print(f"AID {aid:>7} ({cell:20s}) [{source}]: {n_rows:6,} rows, {n_cids:6,} CIDs | {outcomes}")


Data Summary by Cell Line:
----------------------------------------------------------------------
AID     654 (HepG2               ) [NCATS]:  1,393 rows,  1,334 CIDs | {'Inactive': 1355, 'Inconclusive': 23, 'Active': 15}
AID     655 (Jurkat              ) [NCATS]:  1,393 rows,  1,334 CIDs | {'Inactive': 1257, 'Inconclusive': 87, 'Active': 49}
AID     656 (HUV-EC-C            ) [NCATS]:  1,393 rows,  1,334 CIDs | {'Inactive': 1372, 'Inconclusive': 16, 'Active': 5}
AID     657 (SHSY5Y              ) [NCATS]:  1,393 rows,  1,334 CIDs | {'Inactive': 1344, 'Inconclusive': 39, 'Active': 10}
AID     658 (BJ                  ) [NCATS]:  1,393 rows,  1,334 CIDs | {'Inactive': 1359, 'Inconclusive': 28, 'Active': 6}
AID     659 (MRC-5               ) [NCATS]:  1,393 rows,  1,334 CIDs | {'Inactive': 1361, 'Inconclusive': 24, 'Active': 8}
AID     660 (Mesangial           ) [NCATS]:  1,393 rows,  1,334 CIDs | {'Inactive': 1356, 'Inconclusive': 29, 'Active': 8}
AID     661 (SK-N-SH             ) [N

---
## Section 2: Load and Clean Tox21 Caspase Data (AID 1346978, 1346980)

In [5]:
# Load Tox21 caspase assays
print("Loading Tox21 Caspase Assays (HepG2 + CHO-K1)...")
print("="*70)

tox21_aids = ['1346978', '1346980']
for aid in tox21_aids:
    if aid in ASSAY_METADATA:
        df = load_and_clean_assay(aid, ASSAY_METADATA[aid], is_tox21=True)
        if df is not None:
            all_dfs.append(df)

print("="*70)
print(f"\nLoaded {len([d for d in all_dfs if d['source'].iloc[0] == 'Tox21'])} Tox21 assays")
print(f"\nTotal assays loaded: {len(all_dfs)}")

Loading Tox21 Caspase Assays (HepG2 + CHO-K1)...
  AID 1346978 (HepG2               ):  9,667 ->  9,524 rows (removed 143 missing CIDs)
  AID 1346980 (CHO-K1              ):  9,667 ->  9,524 rows (removed 143 missing CIDs)

Loaded 2 Tox21 assays

Total assays loaded: 15


---
## Section 3: Combine All Caspase Data

In [6]:
# Define columns to keep for standardized output
STANDARD_COLS = [
    'PUBCHEM_CID',
    'PUBCHEM_EXT_DATASOURCE_SMILES',
    'PUBCHEM_ACTIVITY_OUTCOME',
    'PUBCHEM_ACTIVITY_SCORE',
    'AID',
    'assay_name',
    'assay_type',
    'cell_line',
    'source',
    'is_counter_screen',
    'biological_process',
    'endpoint_level'
]

def standardize_df(df):
    """Keep only standard columns that exist in the dataframe."""
    available_cols = [c for c in STANDARD_COLS if c in df.columns]
    return df[available_cols].copy()

# Standardize all dataframes
standardized_dfs = [standardize_df(df) for df in all_dfs]

# Combine all
combined_raw = pd.concat(standardized_dfs, ignore_index=True)

print(f"Combined dataset: {len(combined_raw):,} rows")
print(f"Unique CIDs: {combined_raw['PUBCHEM_CID'].nunique():,}")
print(f"\nBy source:")
print(combined_raw['source'].value_counts())
print(f"\nBy cell line:")
print(combined_raw['cell_line'].value_counts())

Combined dataset: 37,157 rows
Unique CIDs: 7,716

By source:
source
Tox21    19048
NCATS    18109
Name: count, dtype: int64

By cell line:
cell_line
HepG2                    10917
CHO-K1                    9524
Jurkat                    1393
HUV-EC-C                  1393
SHSY5Y                    1393
BJ                        1393
MRC-5                     1393
Mesangial                 1393
SK-N-SH                   1393
H-4-II-E                  1393
Hek293                    1393
N2a                       1393
NIH 3T3                   1393
Renal_Proximal_Tubule     1393
Name: count, dtype: int64


In [7]:
# Outcome distribution
print("\nActivity Outcome Distribution:")
print(combined_raw['PUBCHEM_ACTIVITY_OUTCOME'].value_counts())
print("\nBy Cell Line:")
print(combined_raw.groupby('cell_line')['PUBCHEM_CID'].count().sort_values(ascending=False))


Activity Outcome Distribution:
PUBCHEM_ACTIVITY_OUTCOME
Inactive        35492
Inconclusive     1091
Active            574
Name: count, dtype: int64

By Cell Line:
cell_line
HepG2                    10917
CHO-K1                    9524
BJ                        1393
H-4-II-E                  1393
HUV-EC-C                  1393
Hek293                    1393
Jurkat                    1393
MRC-5                     1393
Mesangial                 1393
N2a                       1393
NIH 3T3                   1393
Renal_Proximal_Tubule     1393
SHSY5Y                    1393
SK-N-SH                   1393
Name: PUBCHEM_CID, dtype: int64


---
## Section 4: Deduplication with Merge Logic

**Logic:**
1. If same CID has **different outcomes** → Keep ALL (one row per outcome)
2. If same CID + same outcome → Keep the one with **highest activity score**

In [8]:
print("Deduplication Process")
print("="*70)
print(f"Before deduplication: {len(combined_raw):,} rows")

# Step 1: Sort by CID, outcome, and score (descending)
# This ensures highest score comes first for each CID+outcome combination
combined_sorted = combined_raw.sort_values(
    by=['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE'],
    ascending=[True, True, False]
)

# Step 2: Keep first occurrence of each CID + outcome combination (highest score)
# This preserves DIFFERENT outcomes for the same CID
merged_caspase = combined_sorted.drop_duplicates(
    subset=['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME'],
    keep='first'
).reset_index(drop=True)

print(f"After deduplication: {len(merged_caspase):,} rows")
print(f"Removed: {len(combined_raw) - len(merged_caspase):,} duplicate rows")
print(f"\nUnique CIDs: {merged_caspase['PUBCHEM_CID'].nunique():,}")

Deduplication Process
Before deduplication: 37,157 rows
After deduplication: 8,457 rows
Removed: 28,700 duplicate rows

Unique CIDs: 7,716


In [9]:
# Verify the deduplication logic
print("\nVerification: CIDs with multiple outcome values")
print("-"*50)

# Count CIDs with multiple outcomes
cid_outcome_counts = merged_caspase.groupby('PUBCHEM_CID')['PUBCHEM_ACTIVITY_OUTCOME'].nunique()
multi_outcome_cids = cid_outcome_counts[cid_outcome_counts > 1]

print(f"CIDs with 1 outcome: {(cid_outcome_counts == 1).sum():,}")
print(f"CIDs with 2 outcomes: {(cid_outcome_counts == 2).sum():,}")
print(f"CIDs with 3 outcomes: {(cid_outcome_counts == 3).sum():,}")

# Show example
if len(multi_outcome_cids) > 0:
    example_cid = multi_outcome_cids.index[0]
    print(f"\nExample - CID {example_cid} with multiple outcomes:")
    display(merged_caspase[merged_caspase['PUBCHEM_CID'] == example_cid][['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE', 'AID', 'cell_line']])


Verification: CIDs with multiple outcome values
--------------------------------------------------
CIDs with 1 outcome: 7,079
CIDs with 2 outcomes: 533
CIDs with 3 outcomes: 104

Example - CID 6 with multiple outcomes:


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,AID,cell_line
1,6,Inactive,0,1346978,HepG2
2,6,Inconclusive,10,1346978,HepG2


In [10]:
# Verify highest score was kept
print("\nVerification: Highest score kept for duplicates")
print("-"*50)

# Pick a CID that had duplicates
# Find CIDs that appear multiple times in raw data with same outcome
raw_grouped = combined_raw.groupby(['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME']).size()
duplicate_pairs = raw_grouped[raw_grouped > 1]

if len(duplicate_pairs) > 0:
    example_cid, example_outcome = duplicate_pairs.index[0]
    
    print(f"Example - CID {example_cid} with outcome '{example_outcome}'")
    print(f"\nBEFORE dedup (raw data):")
    before = combined_raw[(combined_raw['PUBCHEM_CID'] == example_cid) & 
                          (combined_raw['PUBCHEM_ACTIVITY_OUTCOME'] == example_outcome)]
    display(before[['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE', 'AID', 'cell_line']])
    
    print(f"\nAFTER dedup (kept highest score):")
    after = merged_caspase[(merged_caspase['PUBCHEM_CID'] == example_cid) & 
                           (merged_caspase['PUBCHEM_ACTIVITY_OUTCOME'] == example_outcome)]
    display(after[['PUBCHEM_CID', 'PUBCHEM_ACTIVITY_OUTCOME', 'PUBCHEM_ACTIVITY_SCORE', 'AID', 'cell_line']])


Verification: Highest score kept for duplicates
--------------------------------------------------
Example - CID 4 with outcome 'Inactive'

BEFORE dedup (raw data):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,AID,cell_line
22985,4,Inactive,0,1346978,HepG2
32509,4,Inactive,0,1346980,CHO-K1



AFTER dedup (kept highest score):


,PUBCHEM_CID,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,AID,cell_line
0,4,Inactive,0,1346978,HepG2


---
## Section 5: Final Summary and Export

In [11]:
print("="*70)
print("FINAL MERGED CASPASE DATASET SUMMARY")
print("="*70)

print(f"\nTotal records: {len(merged_caspase):,}")
print(f"Unique CIDs: {merged_caspase['PUBCHEM_CID'].nunique():,}")

print(f"\n--- By Activity Outcome ---")
for outcome, count in merged_caspase['PUBCHEM_ACTIVITY_OUTCOME'].value_counts().items():
    pct = count / len(merged_caspase) * 100
    print(f"  {outcome}: {count:,} ({pct:.1f}%)")

print(f"\n--- By Source ---")
print(merged_caspase['source'].value_counts())

print(f"\n--- By Cell Line ---")
cell_line_summary = merged_caspase.groupby('cell_line').agg({
    'PUBCHEM_CID': ['count', 'nunique']
}).round(0)
cell_line_summary.columns = ['Total Records', 'Unique CIDs']
print(cell_line_summary.sort_values('Total Records', ascending=False))

FINAL MERGED CASPASE DATASET SUMMARY

Total records: 8,457
Unique CIDs: 7,716

--- By Activity Outcome ---
  Inactive: 7,534 (89.1%)
  Inconclusive: 599 (7.1%)
  Active: 324 (3.8%)

--- By Source ---
source
Tox21    6861
NCATS    1596
Name: count, dtype: int64

--- By Cell Line ---
                       Total Records  Unique CIDs
cell_line                                        
HepG2                           7776         7671
CHO-K1                           400          394
Jurkat                           107          106
NIH 3T3                           35           35
HUV-EC-C                          27           27
H-4-II-E                          24           24
SHSY5Y                            23           23
Hek293                            20           19
SK-N-SH                           17           17
Mesangial                          9            9
Renal_Proximal_Tubule              6            6
BJ                                 5            5
MRC-5            

In [12]:
# Preview merged data
print("\nFirst 20 rows of merged caspase data:")
display(merged_caspase.head(20))


First 20 rows of merged caspase data:


,PUBCHEM_CID,PUBCHEM_EXT_DATASOURCE_SMILES,PUBCHEM_ACTIVITY_OUTCOME,PUBCHEM_ACTIVITY_SCORE,AID,assay_name,assay_type,cell_line,source,is_counter_screen,biological_process,endpoint_level
0,4,CC(CN)O,Inactive,0,1346978,AID_1346978_caspase37_HepG2,caspase,HepG2,Tox21,False,apoptosis,mechanistic
1,6,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,Inactive,0,1346978,AID_1346978_caspase37_HepG2,caspase,HepG2,Tox21,False,apoptosis,mechanistic
2,6,C1=CC(=C(C=C1[N+](=O)[O-])[N+](=O)[O-])Cl,Inconclusive,10,1346978,AID_1346978_caspase37_HepG2,caspase,HepG2,Tox21,False,apoptosis,mechanistic
3,11,C(CCl)Cl,Inactive,0,654,AID_654_caspase3_HepG2,caspase,HepG2,NCATS,False,apoptosis,mechanistic
4,13,C1=CC(=C(C=C1Cl)Cl)Cl,Inactive,0,654,AID_654_caspase3_HepG2,caspase,HepG2,NCATS,False,apoptosis,mechanistic
5,33,C(C=O)Cl,Inactive,0,1346978,AID_1346978_caspase37_HepG2,caspase,HepG2,Tox21,False,apoptosis,mechanistic
6,34,C(CCl)O,Inactive,0,1346978,AID_1346978_caspase37_HepG2,caspase,HepG2,Tox21,False,apoptosis,mechanistic
7,51,C(CC(=O)O)C(=O)C(=O)O,Inactive,0,654,AID_654_caspase3_HepG2,caspase,HepG2,NCATS,False,apoptosis,mechanistic
8,66,C1=CC(=C(C=C1Cl)O)Cl,Inactive,0,654,AID_654_caspase3_HepG2,caspase,HepG2,NCATS,False,apoptosis,mechanistic
9,72,C1=CC(=C(C=C1C(=O)O)O)O,Inactive,0,1346978,AID_1346978_caspase37_HepG2,caspase,HepG2,Tox21,False,apoptosis,mechanistic


In [13]:
# Rename columns for cleaner output
merged_caspase_clean = merged_caspase.rename(columns={
    'PUBCHEM_CID': 'CID',
    'PUBCHEM_EXT_DATASOURCE_SMILES': 'SMILES',
    'PUBCHEM_ACTIVITY_OUTCOME': 'outcome',
    'PUBCHEM_ACTIVITY_SCORE': 'activity_score'
})

print("Columns after renaming:")
print(merged_caspase_clean.columns.tolist())

Columns after renaming:
['CID', 'SMILES', 'outcome', 'activity_score', 'AID', 'assay_name', 'assay_type', 'cell_line', 'source', 'is_counter_screen', 'biological_process', 'endpoint_level']


In [ ]:
# Save merged caspase data
output_path = '../../output/merged_all_caspase_data.csv'
os.makedirs('../../output', exist_ok=True)

merged_caspase_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")
print(f"Shape: {merged_caspase_clean.shape}")
print(f"Columns: {merged_caspase_clean.columns.tolist()}")

---
## Section 6: Additional Analysis - Active Compounds Across Cell Lines

In [16]:
# How many compounds are Active in multiple cell lines?
active_only = merged_caspase_clean[merged_caspase_clean['outcome'] == 'Active']

print("Active Compounds Analysis")
print("="*50)
print(f"Total Active records: {len(active_only):,}")
print(f"Unique Active CIDs: {active_only['CID'].nunique():,}")

# CIDs active in multiple cell lines
active_per_cid = active_only.groupby('CID')['cell_line'].nunique()
print(f"\nCIDs active in:")
print(f"  1 cell line: {(active_per_cid == 1).sum():,}")


Active Compounds Analysis
Total Active records: 324
Unique Active CIDs: 324

CIDs active in:
  1 cell line: 324
